In [1]:
import tensorly as tl

tl.get_backend()

'numpy'

In [2]:
from hoda.hoda import HODA
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from mne.decoding import Scaler
import numpy as np
from sklearn.pipeline import Pipeline
from hoda.hoda import BTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from hoda.classification import SelectF


hoda_params = dict(
    max_iter=128,
    tol=1e-6,
    shrinkage='lw',
    toeplitz=None,
    obj='tr',
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=True,
)

bttda_params = dict(
    verbose=True,
    forward=True,
    extra_train_info=False, 
)

clf = Pipeline([
    ('to_numpy', FunctionTransformer(tl.to_numpy)),
    ('scaler', StandardScaler()),
    ('clf', LDA(shrinkage='auto', solver='lsqr'))
])
deltas = [0] + list(np.geomspace(1e-3,1, 5-1))
clf

Pipeline(steps=[('to_numpy',
                 FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x1479909f0ea0>)),
                ('scaler', StandardScaler()),
                ('clf',
                 LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))])

In [3]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from sklearn.model_selection import StratifiedKFold, GridSearchCV

sfreq = 48
paradigm = P300(resample=sfreq)
datasets = [BNCI2014_008()]

grid_theta = [0, 0.125, 0.25,0.375, 0.5, 0.625, 0.75, 0.875, 1]
grid_n_blocks = list(range(1,16+1))

cv = StratifiedKFold(n_splits=5)

In [ ]:
import pandas as pd
import tensorly as tl
from sklearn.metrics import roc_auc_score
from mne.decoding import Scaler

results = []

for dataset in datasets:
    for subject in dataset.subject_list[:1]:
        data, labels, meta = paradigm.get_data(dataset=dataset, subjects=[subject])
        X = tl.tensor(data)[:100]
        y = labels[:100]
        for fold, (train_idc, test_idc) in enumerate(cv.split(X, y)):
            print(f'fold={fold}')
            scaler = Scaler(scalings='mean', with_mean=True)
            Xs = scaler.fit(X[train_idc])
            Xs = scaler.transform(X)
            
            for theta in grid_theta:
                print(f'theta={theta}')
                hoda_params['theta'] = theta
                bttda = BTTDA(
                    ranks=[None]*max(grid_n_blocks),
                    hoda_params=hoda_params,
                    **bttda_params
                )            
                bttda.fit(Xs[train_idc], y[train_idc])
            
                for n_blocks in grid_n_blocks:
                    print(f'n_blocks={n_blocks}')   
                    Xt = bttda.transform(Xs, n_blocks=n_blocks)
                    X_rec = bttda.inv_transform(Xt, n_blocks=n_blocks)
                    
                    clf.fit(Xt[train_idc], y[train_idc])
                    y_proba_pred = clf.predict_proba(Xt)
                    res = dict(
                        subject = subject,
                        dataset = dataset.code,
                        fold = fold,
                        theta=theta,
                        n_blocks=n_blocks,
                        train_roc_auc = roc_auc_score(y[train_idc], y_proba_pred[train_idc,1]),
                        test_roc_auc = roc_auc_score(y[test_idc], y_proba_pred[test_idc,1]),
                        train_mse = tl.metrics.regression.MSE(X[train_idc], X_rec[train_idc]),
                        test_mse = tl.metrics.regression.MSE(X[test_idc], X_rec[test_idc]),          
                    )
                    results.append(res)

results = pd.DataFrame(results)

In [ ]:
results

In [ ]:
import seaborn as sns
sns.lineplot(data=results, x='n_blocks', y='test_roc_auc', hue='theta', errorbar=None)

In [ ]:
import seaborn as sns
ax = sns.lineplot(data=results, x='n_blocks', y='test_mse', hue='theta', errorbar=None)